In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import json
import pandas as pd
import os
import zipfile
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn as nn
import os, json
from transformers import CLIPModel, AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd

In [ ]:
!pip install -q pytorch-lightning torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.1/823.1 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.2 MB/s eta 0:00:00


In [ ]:
# Set dataset path
dataset_path = "/content/drive/My Drive/licenta/balanced"

# List available files
files = os.listdir(dataset_path)
print("Files in dataset:", files)

Files in dataset: ['test_balanced_questions.json', 'shortAnswer_translated.json', 'testdev_balanced_questions.json', 'train_balanced_questions.json', 'val_balanced_questions.json', 'train_subset_balanced.json', 'val_subset_balanced.json', 'train_top500_filtered.json', 'val_top500_filtered.json', 'test_top500_filtered.json', 'train_top500_subset.json', 'val_top500_subset.json', 'test_top500_subset.json', 'train_top500_100k.json', 'val_top500_balanced.json', 'test_top500_full.json', 'answer_vocab_top500.json', 'train_top100_shared_balanced.json', 'val_top100_shared_balanced.json', 'train_top100_cluster.json', 'val_top100_cluster.json', 'test_top100_shared_balanced.json', 'train_final.json']


In [ ]:
dataset_path = "/content/drive/My Drive/vqa_clean/new_data"

def load_dataset(json_path):
    json_path = os.path.join(dataset_path, json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        data_json = json.load(f)
    df = pd.DataFrame.from_dict(data_json, orient="index")
    df["question"] = df["question"]
    df["answer"] = df["answer_clean"]
    df["imageId"] = df["imageId"]
    return df

train_df = load_dataset("train_all_clean_answer.json")
val_df = load_dataset("val_all_clean_answer.json")
test_df = load_dataset("test_all_clean_answer.json")

In [ ]:
print(f"Train Samples: {len(train_df)}")
print(f"Val Samples:   {len(val_df)}")
print(f"Test Samples:  {len(test_df)}")

Train Samples: 943000
Val Samples:   132062
Test Samples:  12578


In [ ]:
import os
import zipfile

zip_path = "/content/drive/My Drive/licenta/images.zip"  # Adjust if different name
extract_path = "/content/data"  # Folder where you'll unzip

# Create the directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")

Extraction complete!


In [ ]:
import os

image_dir = "/content/data/images"

# 1. List all files in /content/data
all_files = os.listdir(image_dir)

# 2. Filter by image extensions
image_files = [f for f in all_files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# 3. Print stats
print(f"Total files in '{image_dir}': {len(all_files)}")
print(f"Total images (jpg/png/jpeg): {len(image_files)}")
print(f"First 5 images: {image_files[:5]}")

Total files in '/content/data/images': 148854
Total images (jpg/png/jpeg): 148854
First 5 images: ['n439542.jpg', '2319759.jpg', '2331067.jpg', '2371398.jpg', '2381381.jpg']


In [ ]:
extract_path = "/content/data"  # Folder where you'll unzip

In [ ]:
import pandas as pd
import numpy as np

# ──────────────────────────────────────────────────────────────
# 1️⃣  Canonicalise every answer cell to a single clean string
#     • pick first element if the cell is list/tuple/array
#     • strip whitespace, lower-case (optional)
# ──────────────────────────────────────────────────────────────
def canon(cell: object) -> str:
    if pd.isna(cell):
        return None                       # ignore true NaNs
    if isinstance(cell, (list, tuple, np.ndarray)):
        cell = cell[0]                    # keep first synonym
    return str(cell).strip()              # .lower() if you prefer

train_df["answer_c"] = train_df["answer"].map(canon)
val_df["answer_c"]   = val_df["answer"].map(canon)
test_df["answer_c"]  = test_df["answer"].map(canon)

# ──────────────────────────────────────────────────────────────
# 2️⃣  Build mapping from the union of *all three* splits
#     (first-seen order preserved)
# ──────────────────────────────────────────────────────────────
seen = {}
for col in [train_df["answer_c"], val_df["answer_c"], test_df["answer_c"]]:
    for a in col:
        if a is not None and a not in seen:
            seen[a] = len(seen)

answer2idx = seen                            # answer → int
idx2answer = dict(enumerate(seen))           # int    → answer

# Optional <UNK>
UNK_ID = len(answer2idx)
answer2idx["<UNK>"] = UNK_ID
idx2answer[UNK_ID]  = "<UNK>"

print(f"Total unique answers: {len(answer2idx)-1} (+1 UNK)")

# ──────────────────────────────────────────────────────────────
# 3️⃣  Attach integer label to every DataFrame
#     (unknowns mapped to UNK_ID)
# ──────────────────────────────────────────────────────────────
for df in (train_df, val_df, test_df):
    df["answer_id"] = df["answer_c"].map(answer2idx).fillna(UNK_ID).astype(int)


Total unique answers: 1733 (+1 UNK)


In [ ]:
all_answers = sorted(
    set(train_df["answer_clean"])
  | set(val_df  ["answer_clean"])
  | set(test_df ["answer_clean"])
)
answer2idx = {ans: i for i, ans in enumerate(all_answers)}
idx2answer = {i: ans for ans, i in answer2idx.items()}


In [ ]:
print("Number of classes:", len(answer2idx))
print("Example classes:", list(answer2idx.keys())[:10])


Number of classes: 1733
Example classes: ['a da mana', 'a inflori', 'a picta', 'aburi', 'aburit', 'acoperit', 'acoperit de nori', 'acoperiş', 'acvariu', 'adidas']


In [ ]:
!pip install -q pytorch-lightning torchmetrics


In [ ]:
from transformers import ViTModel, AutoTokenizer, AutoModel


In [ ]:
class Config:
    text_model = "dumitrescustefan/bert-base-romanian-cased-v1"
    vision_model = "microsoft/swin-base-patch4-window7-224"
    max_length = 128
    batch_size = 128
    num_epochs = 20  # Increased for early stopping
    learning_rate = 3e-5
    num_classes = 100
    patience = 2  # Early stopping patience
    scheduler_factor = 0.01  # LR reduction factor
    scheduler_threshold = 0.01 # Metric threshold
    min_lr = 1e-6  # Minimum learning rate

In [ ]:
from torchvision.models import resnet101, ResNet101_Weights
from transformers import AutoTokenizer

# ─── 1) Prepare the ResNet101 preprocess pipeline ──────────────────────────
# This is exactly what the docs show:
weights = ResNet101_Weights.DEFAULT
resnet_preprocess = weights.transforms()  # Resize→Crop→ToTensor→Normalize

In [ ]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from timm.data.transforms_factory import create_transform
import requests

In [ ]:
from transformers import AutoModel
from PIL import Image
from timm.data.transforms_factory import create_transform
import requests

In [ ]:
# # 1⃣  install the matching Torch build (already done, keep it)
# !pip install --quiet torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# # 2⃣  pull the pre-built wheel directly from GitHub, then MambaVision (skip deps we already have)
# !pip install -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp311-cp311-linux_x86_64.whl
# !pip install -q mambavision==1.1.0 --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 166.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [torchaudio]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.7/323.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mamba-ssm]


In [ ]:
# # ─── 1) Prepare MambaVision transform ───────────────────────────────
# # Load a dummy model just to grab its config (mean/std/crop)
# mamba = AutoModel.from_pretrained("nvidia/MambaVision-S-1K", trust_remote_code=True)
# mamba.config  # has mean, std, crop_mode, crop_pct
# input_size = (3, 512, 512)
# transform_mamba = create_transform(
#     input_size=input_size,
#     is_training=False,
#     mean=mamba.config.mean,
#     std=mamba.config.std,
#     crop_mode=mamba.config.crop_mode,
#     crop_pct=mamba.config.crop_pct
# )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} 

In [ ]:
def default_image_loader(path_or_url: str):
    if path_or_url.startswith("http"):
        return Image.open(requests.get(path_or_url, stream=True).raw).convert("RGB")
    else:
        return Image.open(path_or_url).convert("RGB")

In [ ]:
# import os
# import random
# from PIL import Image
# import torch
# from torch.utils.data import Dataset
# from torchvision import transforms

# class VQADataset(Dataset):
#     def __init__(self,
#                  dataframe,
#                  image_dir,
#                  answer2idx,
#                  tokenizer,
#                  image_processor,
#                  image_transform=None,
#                  image_augment: transforms.Compose = None,
#                  text_delete_prob: float = 0,
#                  tensor_augment=None,
#                  syn_dict: dict = None,
#                  syn_replace_prob: float = 0):
#         """
#         Args:
#             syn_dict:            {word: [syn1, syn2, ...]} mapping for augmentation
#             syn_replace_prob:    probability to replace a token with a synonym
#         """
#         self.df = dataframe.reset_index(drop=True)
#         self.image_dir = image_dir
#         self.answer2idx = answer2idx
#         self.tokenizer = tokenizer
#         self.image_processor = image_processor
#         self.image_augment = image_augment
#         self.text_delete_prob = text_delete_prob
#         self.syn_dict = syn_dict or {}
#         self.syn_replace_prob = syn_replace_prob
#         self.tensor_augment = tensor_augment


#         if image_transform is None:
#             weights = ResNet101_Weights.DEFAULT
#             image_transform = weights.transforms()        # resize → crop → toTensor → normalise
#         self.image_transform = image_transform

#     def __len__(self):
#         return len(self.df)

#     def _augment_text(self, question: str) -> str:
#         """Existing: randomly delete tokens."""
#         tokens = question.split()
#         if not tokens:
#             return question
#         kept = [t for t in tokens if random.random() > self.text_delete_prob]
#         if not kept:
#             kept = [random.choice(tokens)]
#         return " ".join(kept)

#     def _synonym_replace(self, question: str) -> str:
#         """New: randomly swap tokens for synonyms from syn_dict."""
#         tokens = question.split()
#         out = []
#         for t in tokens:
#             # if we have synonyms for t and coin flip succeeds
#             if t in self.syn_dict and self.syn_dict[t] and random.random() < self.syn_replace_prob:
#                 out.append(random.choice(self.syn_dict[t]))
#             else:
#                 out.append(t)
#         return " ".join(out)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]

#         # --- Image processing (unchanged) ---
#         image_path = os.path.join(self.image_dir, row["imageId"] + ".jpg")
#         image = Image.open(image_path).convert("RGB")
#         if self.image_augment:
#             image = self.image_augment(image)
#         #vit prerprocessor
#         # image = self.image_processor(image, return_tensors="pt")["pixel_values"].squeeze(0)

#         #resnet preprocessor
#         image = self.image_transform(image)  # returns a [3,224,224] tensor


#         if self.tensor_augment:
#             image = self.tensor_augment(image)
#         # --- Text processing w/ augmentation ---
#         question = row["question"]
#         # normalize diacritics
#         question = (question
#                     .replace("ţ", "ț").replace("ş", "ș")
#                     .replace("Ţ", "Ț").replace("Ş", "Ș"))
#         # 1) optionally delete tokens
#         question = self._augment_text(question)
#         # 2) optionally replace with synonyms
#         # question = self._synonym_replace(question)

#         text_enc = self.tokenizer(
#             question,
#             padding="max_length",
#             max_length=Config.max_length,
#             truncation=True,
#             add_special_tokens=True,
#             return_tensors="pt"
#         )

#         label = torch.tensor(self.answer2idx[row["answer"]], dtype=torch.long)


#         return {
#             "image": image,
#             "input_ids": text_enc["input_ids"].squeeze(0),
#             "attention_mask":text_enc["attention_mask"].squeeze(0),
#             "label": label
#         }


# # Example usage:
# # Define image augmentations
# default_img_aug = transforms.Compose([
#     transforms.RandomResizedCrop((224,224), scale=(0.8,1.0)),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
# ])

# # Instantiate dataset
# # train_dataset = VQADataset(train_df, "/path/to/images", answer2idx, tokenizer, image_processor,
# #                            image_augment=default_img_aug, text_delete_prob=0.1)

# # DataLoader remains unchanged:
# # train_loader = DataLoader(train_dataset, batch_size=..., shuffle=True, num_workers=4, pin_memory=True)



In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class VQADataset(Dataset):
    def __init__(self,
                 dataframe,
                 image_dir,
                 answer2idx,
                 tokenizer,
                 image_processor,
                 image_transform=None,
                 image_augment=None,
                 text_delete_prob: float = 0,
                 question_bt_prob: float = 0.0,   # ← new
                 tensor_augment=None,
                 syn_dict: dict = None,
                 syn_replace_prob: float = 0 ,
                 backtranslation_prob: float = 0.0):
        self.df                = dataframe.reset_index(drop=True)
        # …
        self.text_delete_prob  = text_delete_prob
        self.question_bt_prob  = question_bt_prob   # ← store it
        # …
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.answer2idx = answer2idx
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.image_augment = image_augment
        self.text_delete_prob = text_delete_prob
        self.syn_dict = syn_dict or {}
        self.syn_replace_prob = syn_replace_prob
        self.tensor_augment = tensor_augment
        self.backtranslation_prob = backtranslation_prob

        # if image_transform is None:
        #     weights = ResNet101_Weights.DEFAULT
        #     image_transform = weights.transforms()        # resize → crop → toTensor → normalise
        self.image_transform = image_transform


    def __len__(self):
        return len(self.df)

    def _augment_text(self, question: str) -> str:
        """Existing: randomly delete tokens."""
        tokens = question.split()
        if not tokens:
            return question
        kept = [t for t in tokens if random.random() > self.text_delete_prob]
        if not kept:
            kept = [random.choice(tokens)]
        return " ".join(kept)

    def _synonym_replace(self, question: str) -> str:
        """New: randomly swap tokens for synonyms from syn_dict."""
        tokens = question.split()
        out = []
        for t in tokens:
            # if we have synonyms for t and coin flip succeeds
            if t in self.syn_dict and self.syn_dict[t] and random.random() < self.syn_replace_prob:
                out.append(random.choice(self.syn_dict[t]))
            else:
                out.append(t)
        return " ".join(out)


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # … image loading + augmentation as before …
         # --- Image processing (unchanged) ---
        image_path = os.path.join(self.image_dir, row["imageId"] + ".jpg")
        image = Image.open(image_path).convert("RGB")
        if self.image_augment:
            image = self.image_augment(image)
        #vit prerprocessor
        # image = self.image_processor(image, return_tensors="pt")["pixel_values"].squeeze(0)

        #resnet preprocessor
        # image = self.image_transform(image)  # returns a [3,224,224] tensor
         # 2) HF processor → {"pixel_values": Tensor[1,3,H,W], ...}
        proc_out     = self.image_processor(image, return_tensors="pt")
        pixel_values = proc_out["pixel_values"].squeeze(0)  # now [3,H,W]

        if self.tensor_augment:
            pixel_values = self.tensor_augment(pixel_values)

        # ─── pick original vs. back-translated ──────────────────────────
        q_orig = row["question"]
        q_bt   = row.get("question_bt", "")
        # if we have a BT version and hit the flip-of-the-coin, use it:
        if q_bt and random.random() < self.backtranslation_prob:
            question = q_bt
        else:
            question = q_orig

        # ─── continue your text augment pipeline ───────────────────────
        # 1) normalize diacritics…
        question = (question
                    .replace("ţ", "ț").replace("ş", "ș")
                    .replace("Ţ", "Ț").replace("Ş", "Ș"))
        # 2) random deletion…
        question = self._augment_text(question)

        # 3) optional synonym replace…
        # …
        question = self._synonym_replace(question)

        # ─── tokenize + return ──────────────────────────────────────────
        text_enc = self.tokenizer(
            question,
            padding="max_length",
            max_length=Config.max_length,
            truncation=True,
            return_tensors="pt"
        )
        label = torch.tensor(self.answer2idx[row["answer"]], dtype=torch.long)
        return {
            "image":          pixel_values,
            "input_ids":      text_enc["input_ids"].squeeze(0),
            "attention_mask": text_enc["attention_mask"].squeeze(0),
            "label":          label
        }



# Example usage:
# Define image augmentations
default_img_aug = transforms.Compose([
    transforms.RandomResizedCrop((224,224), scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
])

# Instantiate dataset
# train_dataset = VQADataset(train_df, "/path/to/images", answer2idx, tokenizer, image_processor,
#                            image_augment=default_img_aug, text_delete_prob=0.1)

# DataLoader remains unchanged:
# train_loader = DataLoader(train_dataset, batch_size=..., shuffle=True, num_workers=4, pin_memory=True)



In [ ]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

# 1) PIL-only augmentations
strong_pil_aug = transforms.Compose([
    transforms.RandomResizedCrop(
        size=(224, 224),
        scale=(0.5, 1.0),
        ratio=(0.75, 1.33),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.RandomAffine(
            degrees=15,
            translate=(0.05, 0.05),
            shear=5,
            interpolation=InterpolationMode.BICUBIC,
            fill=128
        )
    ], p=0.5),
    transforms.RandomApply([
        transforms.RandomPerspective(distortion_scale=0.4, p=1.0)
    ], p=0.3),
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03
    ),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
])

# 2) Tensor-only augmentations
tensor_aug = transforms.Compose([
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value='random'
    )
])


In [ ]:
import torch
import torch.nn as nn
from transformers import ViTModel, AutoModel

class EarlyFusionModel(nn.Module):
    def __init__(self, unfreeze_last_n: int = 3):
        """
        unfreeze_last_n – how many *last* transformer blocks to unfreeze
                          in both the vision and text encoders.
        """
        super().__init__()

        # ---------- Encoders ----------
        self.vision_encoder = ViTModel.from_pretrained(Config.vision_model)
        self.text_encoder   = AutoModel.from_pretrained(Config.text_model)

        # ---------- 1) Freeze all parameters ----------
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        for p in self.text_encoder.parameters():
            p.requires_grad = False

        # ---------- 2) Unfreeze last `n` blocks in each encoder ----------
        for block in self.vision_encoder.encoder.layer[-unfreeze_last_n:]:
            for p in block.parameters():
                p.requires_grad = True
        for block in self.text_encoder.encoder.layer[-unfreeze_last_n:]:
            for p in block.parameters():
                p.requires_grad = True

        # ---------- Classifier Head ----------
        vision_hidden = self.vision_encoder.config.hidden_size
        text_hidden   = self.text_encoder.config.hidden_size
        fused_dim     = vision_hidden + text_hidden

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, Config.num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        # image: tensor of shape [B, 3, H, W]
        img_out = self.vision_encoder(pixel_values=image)
        img_cls = img_out.last_hidden_state[:, 0, :]

        txt_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        txt_cls = txt_out.last_hidden_state[:, 0, :]

        fused = torch.cat([img_cls, txt_cls], dim=1)
        return self.classifier(fused)


###RESNET MODEL


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet101, ResNet101_Weights
from transformers import AutoModel

class EarlyFusionResNetModel(nn.Module):
    """
    Vision branch: ResNet-101 (ImageNet weights, fc removed, 2048-D global-pool features)
    Text   branch: any HF encoder (e.g. BERT/RoBERTa)

    unfreeze_last_n  →  how many *last* ResNet blocks (layer4, layer3…) **and**
                         how many last transformer blocks to fine-tune.
    """
    def __init__(self,
                 unfreeze_last_n: int = 2,        # 0 = keep both encoders frozen (“feature extractor” mode)
                 text_model_name: str = None):
        super().__init__()

        # -------- Vision encoder (ResNet-101) -------------------------------
        weights  = ResNet101_Weights.DEFAULT
        backbone = resnet101(weights=weights)

        self.vision_encoder = nn.Sequential(*list(backbone.children())[:-1])   # conv→…→avgpool
        vision_hidden = backbone.fc.in_features      # 2048

        # -------- Text encoder (HF) -----------------------------------------
        if text_model_name is None:
            text_model_name = Config.text_model
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_hidden = self.text_encoder.config.hidden_size     # 768 for BERT-base

        # -------- Freeze everything -----------------------------------------
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        for p in self.text_encoder.parameters():
            p.requires_grad = False

        # -------- Unfreeze last *n* blocks ----------------------------------
        # ResNet “blocks” are layer4, layer3, layer2, layer1  (in that order)
        res_blocks = [self.vision_encoder[-2],   # layer4
                      self.vision_encoder[-3],   # layer3
                      self.vision_encoder[-4],   # layer2
                      self.vision_encoder[-5]]   # layer1
        for blk in res_blocks[:unfreeze_last_n]:
            for p in blk.parameters():
                p.requires_grad = True

        # Transformer blocks
        for blk in self.text_encoder.encoder.layer[-unfreeze_last_n:]:
            for p in blk.parameters():
                p.requires_grad = True

        # -------- Classifier head -------------------------------------------
        fused_dim = vision_hidden + text_hidden         # 2048 + 768 = 2816
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, Config.num_classes)
        )

    # ------------------------------------------------------------------------
    def forward(self, image, input_ids, attention_mask):
        """
        image         – [B, 3, 224, 224] already passed through
                        ResNet-101 weights.transforms() (your dataset does this)
        input_ids     – [B, L]
        attention_mask– [B, L]
        """
        # CNN branch
        v = self.vision_encoder(image)          # [B, 2048, 1, 1]
        v = v.flatten(1)                        # [B, 2048]

        # Transformer branch
        t = self.text_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            ).last_hidden_state[:, 0, :]        # [B, 768]

        fused = torch.cat([v, t], dim=1)        # [B, 2816]
        return self.classifier(fused)


##Mamba vision model

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

MAMBA_ID = "nvidia/MambaVision-S-1K"   # (S/B/L)-1K checkpoints

class EarlyFusionMambaModel(nn.Module):
    """
    MambaVision image encoder  +  Romanian text encoder
    CLS vectors are concatenated  →  MLP classifier.
    The whole vision backbone is trainable; text is frozen except the
    last `unfreeze_last_n` transformer blocks.
    """

    def __init__(self, unfreeze_last_n: int = 0):
        super().__init__()

        # ── vision ─────────────────────────────────────────────────────
        self.vision_encoder = AutoModel.from_pretrained(
            MAMBA_ID, trust_remote_code=True         # returns (avg_pool, feats)
        )
        # v_cfg = self.vision_encoder.config
        # v_dim = (getattr(v_cfg, "hidden_size", None) or
        #          getattr(v_cfg, "d_model", None)    or
        #          getattr(v_cfg, "embed_dim", None))

        v_dim = self.vision_encoder.model.head.in_features

        # ── text ───────────────────────────────────────────────────────
        self.text_encoder = AutoModel.from_pretrained(Config.text_model)
        for p in self.text_encoder.parameters():
            p.requires_grad = False
        if unfreeze_last_n > 0:
            for blk in self.text_encoder.encoder.layer[-unfreeze_last_n:]:
                for p in blk.parameters():
                    p.requires_grad = True
        t_dim = self.text_encoder.config.hidden_size

        # ── head ───────────────────────────────────────────────────────
        fused_dim = v_dim + t_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, Config.num_classes)
        )

    # ── forward ────────────────────────────────────────────────────────
    def forward(self, image, input_ids, attention_mask):
        img_vec, _ = self.vision_encoder(image)     # [B, v_dim]

        txt_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        txt_vec = txt_out.last_hidden_state[:, 0, :]                    # [B, t_dim]

        fused = torch.cat([img_vec, txt_vec], dim=1)                    # [B, v+t]
        return self.classifier(fused)


#Swin

In [ ]:
import torch
import torch.nn as nn
from transformers import SwinModel, AutoImageProcessor, AutoModel

class EarlyFusionSwinModel(nn.Module):
    """
    Same as before, but now explicitly uses SwinModel.last_hidden_state
    (shape [B, num_patches, hidden_size]) and mean-pools it into [B, hidden_size].
    """
    def __init__(self, unfreeze_last_n: int = 2):
        super().__init__()
        swin_ckpt = "microsoft/swin-base-patch4-window7-224"
        # swin_ckpt = "microsoft/swin-tiny-patch4-window7-224"
        # self.image_processor = AutoImageProcessor.from_pretrained(swin_ckpt)
        self.vision_encoder  = SwinModel.from_pretrained(swin_ckpt)
        v_dim = self.vision_encoder.config.hidden_size

        self.text_encoder = AutoModel.from_pretrained(Config.text_model)
        t_dim = self.text_encoder.config.hidden_size

        # freeze both
        for p in self.vision_encoder.parameters(): p.requires_grad = False
        for p in self.text_encoder.parameters():   p.requires_grad = False

        # optionally unfreeze last N Swin *stages*
        if unfreeze_last_n > 0:
            swin_layers = list(self.vision_encoder.encoder.layers)
            for stage in swin_layers[-unfreeze_last_n:]:
                for p in stage.parameters():
                    p.requires_grad = True

        # optionally unfreeze last N text *blocks*
        for blk in self.text_encoder.encoder.layer[-unfreeze_last_n:]:
            for p in blk.parameters(): p.requires_grad = True

        # fusion head
        fused_dim = v_dim + t_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, Config.num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        # image: [B,3,H,W] already created via image_processor in your Dataset
        v_out = self.vision_encoder(image, return_dict=True)
        # switch to last_hidden_state + mean pool
        v_tokens = v_out.last_hidden_state           # [B, num_patches, v_dim]
        v_cls    = v_tokens.mean(dim=1)              # [B, v_dim]

        t_out = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        t_cls = t_out.last_hidden_state[:, 0, :]     # [B, t_dim]

        fused = torch.cat([v_cls, t_cls], dim=1)     # [B, v_dim + t_dim]
        return self.classifier(fused)


In [ ]:
# Training Utilities
def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "label": torch.stack([x["label"] for x in batch])
    }

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

# ─── 1) Compute class weights from your train split ────────────────────────────
# Map each cleaned answer string to its integer ID
train_labels = train_df["answer_clean"].map(answer2idx).values
num_classes  = len(answer2idx)

# Count examples per class
class_counts = np.bincount(train_labels, minlength=num_classes)

# Inverse‐frequency: weight[c] = N / (C * count[c])
# where N = total samples, C = num_classes
N = train_labels.shape[0]
C = num_classes
eps = 1e-6
weights = N / (C * (class_counts + eps))

# (Optional) Clamp weights to avoid extremes
weights = np.clip(weights, a_min=0.5, a_max=5.0)

# Tensor on GPU (or CPU)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weight_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

In [ ]:
# ─── 2) Focal Loss class ─────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.weight    = weight
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # 1) Standard CE per‐sample
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        # 2) Compute pt = exp(−CE)
        pt = torch.exp(-ce)
        # 3) Focal term
        focal = (1 - pt) ** self.gamma * ce
        # 4) Reduction
        if self.reduction == "mean":
            return focal.mean()
        elif self.reduction == "sum":
            return focal.sum()
        else:
            return focal

In [ ]:
# 1⃣ Install Lightning (run once in Colab)

# 2⃣ Imports
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import (
    ModelCheckpoint, RichProgressBar, EarlyStopping
)
import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics import Accuracy
from torchmetrics.classification.accuracy import MulticlassAccuracy  # <—
from sklearn.metrics import classification_report

# 3⃣ LightningModule
class LitLateFusion(pl.LightningModule):
    def __init__(self, learning_rate, weight_decay,
                 unfreeze_last_n, scheduler_factor, scheduler_patience,
                 scheduler_threshold, min_lr, num_classes, idx2label: dict, weight_tensor: torch.Tensor):
        super().__init__()
        # save hparams for logging/checkpoint naming
        self.save_hyperparameters(ignore=["idx2label", "weight_tensor"])
        self.weight_tensor = weight_tensor

        # your actual model
        self.backbone = EarlyFusionSwinModel(unfreeze_last_n)
        self.register_buffer("loss_weight", weight_tensor)
      #   self.criterion = FocalLoss(
      #     weight=weight_tensor,
      #     gamma=2.0,
      #     reduction="mean"
      # )
        self.criterion = nn.CrossEntropyLoss(
              # weight=weight_tensor,      # if you’re still using class weights
              label_smoothing=0.1       # <-- this is the label-smoothing parameter
              # reduction="mean"
          )

        # metrics
        self.train_acc = MulticlassAccuracy(num_classes=num_classes)
        self.val_acc   = MulticlassAccuracy(num_classes=num_classes)
        self.idx2label = idx2label
        self.test_preds  = []
        self.test_labels = []

    def forward(self, image, input_ids, attention_mask):
        return self.backbone(image, input_ids, attention_mask)

    def training_step(self, batch, batch_idx):
        # exactly the per-batch code from your train_epoch
        inputs = {k: v for k,v in batch.items() if k!="label"}
        labels = batch["label"]

        outputs = self(**inputs)
        loss    = self.criterion(outputs, labels)

        preds = outputs.argmax(dim=1)
        acc   = (preds == labels).float().mean()

        self.log("train/loss", loss,  on_epoch=True, prog_bar=True)
        self.log("train/acc",  acc,  on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        # 1) Unpack
        inputs = {k: v for k, v in batch.items() if k != "label"}
        labels = batch["label"]

        # 2) Forward + loss
        logits = self(**inputs)
        loss   = self.criterion(logits, labels)

        # 3) Compute batch accuracy
        preds = torch.argmax(logits, dim=1)
        acc   = (preds == labels).float().mean()

        # 4) Log to TensorBoard and progress bar
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/acc",  acc,  on_step=False, on_epoch=True, prog_bar=True)


        # # (optional) return for a custom epoch‐end hook
        # return {"val_loss": loss, "val_acc": acc}

    def test_step(self, batch, batch_idx):
        # 1) forward
        logits = self(**{k: v for k, v in batch.items() if k != "label"})
        preds  = logits.argmax(dim=1)
        labels = batch["label"]

        # 2) collect for later
        self.test_preds.append(preds.cpu())
        self.test_labels.append(labels.cpu())

        # 3) (optional) log batch‐level test loss/acc
        loss = self.criterion(logits, labels)
        acc  = (preds == labels).float().mean()
        self.log("test/loss", loss, on_epoch=True, prog_bar=True)
        self.log("test/acc",  acc,  on_epoch=True, prog_bar=True)

    def on_test_epoch_end(self):
        # concatenate all preds & labels as before
        preds  = torch.cat(self.test_preds).cpu().numpy()
        labels = torch.cat(self.test_labels).cpu().numpy()

        # only the classes you care about
        labels_present = sorted(set(labels) | set(preds))
        target_names   = [self.idx2label[i] for i in labels_present]

        # produce a dict instead of a str
        report_dict = classification_report(
            labels,
            preds,
            labels=labels_present,
            target_names=target_names,
            zero_division=0,
            output_dict=True     # ← return as dict
        )

        # ❶ print to console
        print("\n🧾 TEST SET CLASSIFICATION REPORT\n")
        print(classification_report(
            labels, preds,
            labels=labels_present,
            target_names=target_names,
            zero_division=0
        ))

        # ❷ save to JSON (change path as you like)
        out_path = "/content/drive/MyDrive/licenta/big_data/reports/test_report_early_big_data_swin.json"
        with open(out_path, "w", encoding="utf-8") as fp:
            json.dump(report_dict, fp, ensure_ascii=False, indent=2)
        print(f"✅ Classification report saved to {out_path}")

        # clear buffers for potential re-test
        self.test_preds.clear()
        self.test_labels.clear()

    # (Optional) if you want to aggregate or print a custom summary:
    # def validation_epoch_end(self, outputs):
    #     # outputs is a list of the dicts returned by validation_step
    #     avg_loss = torch.stack([x["val_loss"] for x in outputs]).mean()
    #     avg_acc  = torch.stack([x["val_acc"]  for x in outputs]).mean()
    #     # you could log again or print here, but Lightning will already show the metrics
    #     # by virtue of self.log(on_epoch=True) above.

    def configure_optimizers(self):
        # optim_ = optim.AdamW(
        #     self.parameters(),
        #     lr=self.hparams.learning_rate,
        #     weight_decay=self.hparams.weight_decay
        # )
        optim_ = torch.optim.AdamW(
            (p for p in self.backbone.parameters() if p.requires_grad),
            lr=self.hparams.learning_rate, weight_decay=self.hparams.weight_decay)
        sched = {
            "scheduler": optim.lr_scheduler.ReduceLROnPlateau(
                optim_,
                mode="max",
                factor=self.hparams.scheduler_factor,
                patience=self.hparams.scheduler_patience,
                threshold=self.hparams.scheduler_threshold,
                min_lr=self.hparams.min_lr,
                verbose=True
            ),
            "monitor": "val/acc",
            "interval": "epoch",
            "frequency": 1
        }
        return [optim_], [sched]


In [ ]:
Config.num_classes = len(answer2idx)  # Update based on actual data
print(f"Number of classes: {Config.num_classes}")

Number of classes: 1733


In [ ]:
print(Config.vision_model)

microsoft/swin-base-patch4-window7-224


In [ ]:
from transformers import AutoImageProcessor


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Config.text_model)
image_processor = AutoImageProcessor.from_pretrained(
    Config.vision_model,
    use_fast=True          # ← one flag
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from torch.utils.data import WeightedRandomSampler

train_labels = train_df["answer_clean"].map(answer2idx).values
class_counts = np.bincount(train_labels, minlength=len(answer2idx))
sample_weights = 1.0 / (class_counts[train_labels] + 1e-6)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
syn_dict = {
    # — Function words & stop-words (keep empty) —
    "ce":       [],
    "în":       [],
    "de":       [],
    "pe":       [],
    "sau":      [],
    "se":       [],
    "din":      [],
    "a":        [],
    "și":       [],
    "o":        [],
    "un":       [],
    "la":       [],
    "cu":       [],
    "că":       [],
    "să":       [],
    "in":       [],

    # — Verbs —
    "este":     ["e", "se află", "se găsește"],
    "există":  ["se află", "se găsește"],
    "află":     ["descoperă", "găsește"],
    "poartă":   ["ține", "transportă"],
    "arată":    ["prezintă", "demonstrează"],
    "vedeți":   ["observați", "priviți"],
    "ține":     ["susține", "deține"],
    "pare":     ["se pare", "se înfățișează"],
    "face":     ["realizează", "execută"],
    "făcut":    ["realizat", "efectuat"],
    "numește":  ["denumește", "intitulează"],
    "crezi":    ["credeți", "considerați"],
    "uită":     ["neglijează", "trece cu vederea"],
    "mănâncă":  ["consumează", "degustă"],
    "joacă":    ["se joacă", "practică"],
    "merge":    ["se deplasează", "parcurge"],

    # — Interrogative / directional —
    "care":     ["ce", "care anume"],
    "cine":     ["ce persoană", "care persoană"],
    "cum":      ["în ce fel", "cum anume"],
    "unde":     ["în ce loc", "unde anume"],
    "cât":      ["cât de", "cât anume"],
    "deasupra": ["deasupra", "peste"],
    "jos":      ["sub", "în jos"],
    "sus":      ["deasupra", "în sus"],
    "dreapta":  ["partea dreaptă", "latura dreaptă"],
    "stânga":   ["partea stângă", "latura stângă"],
    "sub":      ["dedesubt", "sub nivelul"],

    # — Nouns & adjectives —
    "culoare":  ["nuanță", "ton"],
    "culoarea": ["nuanța", "tonul"],
    "fel":      ["mod", "tip"],
    "partea":   ["segmentul", "sectorul"],
    "parte":    ["secțiune", "segment"],
    "imaginii": ["fotografiei", "instantanee"],
    "imagine":  ["fotografie", "ilustrație"],
    "fotografie":["imagine", "instantaneu"],
    "fotografiei":["imagină", "instantaneu"],
    "mobilier": ["mobilă", "mobileră"],
    "mobilierul":["mobilă", "mobileră"],
    "mobil":    ["dispozitiv mobil", "telefon"],
    "transport":[],  # skip if not in list
    "vehicul":  ["autovehicul", "automobil"],
    "vehiculul":["autovehiculul", "autoturismul"],
    "mașina":   ["autoturism", "automobil"],
    "mașini":   ["automobile", "vehicule"],
    "mașinii":  ["autoturismului", "automobilului"],
    "bicicleta":["bicicletă", "vehicul cu două roți"],
    "tren":     ["trenuleț", "garnitură feroviară"],
    "autobuz":  ["autocar", "microbuz"],
    "autobuze": ["autocare", "microbuze"],

    "animal":   ["viețuitoare", "fiară"],
    "animalul": ["viețuitoarea", "fiara"],
    "păsare":   [],  # skip if not in list
    "câine":    ["câinele", "canin"],
    "pisica":   ["pisica", "felină"],
    "copilul":  ["copil", "prichindel"],
    "băiatul":  ["tânărul", "băiețelul"],
    "fata":     ["fată", "fetiță"],
    "femeia":   ["doamna", "domnișoara"],
    "femeii":   ["doamnei", "domnișoarei"],
    "bărbatul": ["domnul", "masculinul"],
    "bărbatului":["domnului", "masculinului"],

    "omul":     ["individul", "persoana"],
    "persoana": ["individul", "subiectul"],
    "persoanei":["individului", "subiectului"],

    "masa":     ["tavă", "suprafață plată"],
    "mesei":    ["tăvii", "suprafaței"],

    "scaun":    ["fotoliu", "canapea"],
    "scaune":   ["fotolii", "mese și scaune"],
    "scaunul":  ["fotoliul", "canapeaua"],
    "scaunului":["fotoliului", "canapelei"],

    "pat":      ["paturile", "spital"],  # adjust as needed
    "perna":    ["perna", "pătură mică"],
    "perne":    ["perne", "ornament"],

    "telefonul": ["mobil", "smartphone"],
    "dispozitiv": ["aparatură", "echipament"],
    "dispozitivul":["aparatura", "echipamentul"],

    "legumă":   ["verdețuri", "vegetală"],
    "legumele": ["verdețuri", "vegetalele"],

    "farfurie": ["tavă", "farfurioară"],
    "farfuria": ["tăvița", "farfurioara"],

    "farfurii": ["tăvițe", "farfurioare"],

    "ceasul":   ["orologiu", "cronometru"],
    "ceas":     ["orologiu", "cronometru"],

    "oglinda":  ["oglindă", "reflector"],
    "oglinda":  ["oglindă", "reflector"],

    "loc":      ["spațiu", "zona"],
    "interior": ["înăuntru", "în interior"],
    "scenă":    ["decor", "fundal"],

    "culori":   ["nuanțe", "tonuri"],

    "verde":    ["verde crud", "verde închis"],
    "negru":    ["negru intens", "negru absolut"],
    "gri":      ["gri deschis", "gri închis"],
    "albastru": ["azuriu", "cerulean"],
    "albastră": ["azurie", "ceruleană"],
    "roșu":     ["roșu aprins", "roșu intens"],
    "roșie":    ["roșie aprinsă", "roșie intensă"],
    "alb":      ["alb pur", "alb imaculat"],
    "albe":     ["albe curate", "albe imaculate"],
    "albă":     ["albă pură", "albă imaculată"],

    "maro":     ["brun", "căprui"],
    "portocaliu":["portocaliu aprins", "portocaliu intens"],
    "galben":   ["galben luminos", "galben strălucitor"],
    "galbenă":  ["galbenă luminoasă", "galbenă strălucitoare"],

    "roz":      ["roz pal", "roz deschis"],

    # — Spatial & demonstratives —
    "această":  ["prezentă", "actuală"],
    "aceeași":  ["identică", "aceeași exact"],
    "acesta":   ["acest", "de față"],
    "acea":     ["aceea", "acea anume"],

    "aceeași":  ["identică", "aceeași exact"],

    # — Quantifiers & adjectives —
    "mare":     ["imens", "extins"],
    "mic":      ["minuscul", "restrâns"],
    "mică":     ["minusculă", "restrânsă"],
    "mici":     ["miniaturale", "restrânse"],

    "diferită": ["distinctă", "altă"],

    # — Transport & objects —
    "bicicleta":["bicicletă", "vehicul cu două roți"],

    "autobuz":  ["autocar", "microbuz"],
    "tren":     ["garnitură feroviară", "trenuleț"],
    "camion":   ["tir", "autotractor"],
    "camionul": ["tirul", "autotractorul"],

    "skateboard": ["placă", "skate"],

    "racheta":  ["lanternă", "dronă"],  # adjust if mismatched

    # — Containers & parts —
    "cutia":    ["recipient", "căsuță"],
    "cutiei":   ["recipientului", "căsuței"],

    "coșul":    ["hamar", "coș"],
    "coșuri":   ["hamare", "coșuri"],

    "raftul":   ["etagieră", "policopi"],

    "dulapul":  ["șifonier", "comoda"],
    "dulap":    ["șifonier", "comodă"],

    "prosopul": ["șervețel", "prosop de baie"],

    # — Clothing —
    "cămașa":   ["bluză", "tunică"],
    "cămașă":  ["bluză", "tunică"],
    "jacheta":  ["haină", "geacă"],
    "pantaloni":["blugi", "îmbrăcăminte de jos"],
    "pantalonii":["blugii", "pantalonii denim"],

    "tricoul":  ["bluză", "tricou sport"],

    "îmbrăcăminte":["veșminte", "haine"],

    # — Food & kitchen —
    "mâncare":  ["hrană", "alimentație"],
    "mâncarea":["hrană", "alimentul"],
    "pizza":    ["plăcintă italiană", "pizză"],

    # — Household & misc —
    "sticlă":   ["recipient de sticlă", "pahar"],
    "sticla":   ["recipientul de sticlă", "paharul"],

    "metal":    ["aliaj metalic", "fier"],
    "plastic":  ["material plastic", "polimer"],

    "perete":   ["element vertical", "zid"],
    "peretele": ["elementul vertical", "zidul"],

    "podea":    ["sol", "planșeu"],
    "podeaua":  ["solul", "planșeul"],

    "fereastră":["geam", "ferestră"],   # adjust spelling if needed
    "ferestre":["geamuri", "ferestrărie"],

    "umbrelă":  ["parasol", "pălărie de ploaie"],
    "umbrele":  ["parasole", "pălării de ploaie"],

    # — Colors repeated handled above —
    # — Anything else (leave empty) —
}

# Example access:
print(syn_dict["culoare"])   # → ['nuanță', 'ton']
print(syn_dict["dreapta"])   # → ['partea dreaptă', 'latura dreaptă']


['nuanță', 'ton']
['partea dreaptă', 'latura dreaptă']


In [ ]:
# weights      = ResNet101_Weights.DEFAULT
# resnet_pre   = weights.transforms()          # always the same

In [ ]:
# # ─── 1) Prepare MambaVision transform ───────────────────────────────
# # Load a dummy model just to grab its config (mean/std/crop)
# mamba = AutoModel.from_pretrained("nvidia/MambaVision-S-1K", trust_remote_code=True)
# mamba.config  # has mean, std, crop_mode, crop_pct
# input_size = (3, 224, 224)
# transform_mamba = create_transform(
#     input_size=input_size,
#     is_training=False,
#     mean=mamba.config.mean,
#     std=mamba.config.std,
#     crop_mode=mamba.config.crop_mode,
#     crop_pct=mamba.config.crop_pct
# )

In [ ]:
# 2) Create train/val/test datasets
train_dataset = VQADataset(
    dataframe       = train_df,
    image_dir       = image_dir,
    answer2idx      = answer2idx,
    tokenizer       = tokenizer,
    image_processor = image_processor,
    image_transform = image_processor,
    image_augment   = strong_pil_aug,    # ✔ augmentation ON here
    tensor_augment=None,
    text_delete_prob= 0.2,                 # ✔ text aug ON here
    syn_dict=syn_dict,
    syn_replace_prob=0.4,   # 20% replace chance per token
    backtranslation_prob=0.4,    # 20% of questions get back-translated
)

val_dataset = VQADataset(
    dataframe       = val_df,
    image_dir       = image_dir,
    answer2idx      = answer2idx,
    tokenizer       = tokenizer,
    image_transform = image_processor,
    image_processor = image_processor,
    tensor_augment = None,
    image_augment   = None,               # ❌ no image aug
    text_delete_prob= 0.0,                 # ❌ no text aug
    syn_dict=None,
    syn_replace_prob=0   # 20% replace chance per token
)

test_dataset = VQADataset(
    dataframe       = test_df,
    image_dir       = image_dir,
    answer2idx      = answer2idx,
    tokenizer       = tokenizer,
    image_transform = image_processor,
    image_processor = image_processor,
    tensor_augment = None,
    image_augment   = None,               # ❌ no image aug
    text_delete_prob= 0.0,                # ❌ no text aug
    syn_dict=None,
    syn_replace_prob=0   # 20% replace chance per token
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=Config.batch_size,  num_workers=10, pin_memory=True, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=Config.batch_size, shuffle=False, num_workers=10, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=Config.batch_size, shuffle=False, num_workers=10, pin_memory=True)

In [ ]:
import torch

# 1) Let cuDNN auto-tuner pick the fastest conv algorithms for your input sizes
torch.backends.cudnn.benchmark = True

# 2) Enable TF32 on Ampere+ GPUs for faster matmuls (slightly lower precision)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32        = True

In [ ]:
import glob

# adjust this to wherever you pointed your checkpoints
base_dir = "/content/drive/MyDrive/tb_runs/early-swin"

# recursively find all .ckpt files
ckpts = glob.glob(f"{base_dir}/**/*.ckpt", recursive=True)
print("Found checkpoints:\n", "\n".join(ckpts))

Found checkpoints:
 /content/drive/MyDrive/tb_runs/early-swin/early-swin-best-all-data-attention.ckpt


In [ ]:
# pick the one you want
best_ckpt = ckpts[0]   # or however you identify it
print("Will resume from:", best_ckpt)

Will resume from: /content/drive/MyDrive/tb_runs/early-swin/early-swin-best-all-data-attention.ckpt


In [ ]:


# 5⃣ LightningLogger & Callbacks
logger = TensorBoardLogger(
    save_dir="/content/drive/MyDrive/tb_runs",
    name="early-swin"
)

checkpoint_cb = ModelCheckpoint(
    monitor="val/acc",                # the metric to watch
    mode="max",                       # we want to maximize val/acc
    dirpath="/content/drive/MyDrive/tb_runs/early-swin",
    filename="early-swin-best-all-data-attention",       # this will become latefusion-best.ckpt
    save_top_k=1,                     # keep only the best‐ever checkpoint
    save_last=False,                  # don’t save a “last.ckpt”
    save_on_train_epoch_end=True     # only save when val/acc improves
)

earlystop_cb = EarlyStopping(
    monitor="val/acc",
    mode="max",
    patience=3,
    verbose=True
)

# best_ckpt = checkpoint_cb.best_model_path
# print("Resuming from:", best_ckpt)

# Optional: nicer progress bar
progress_cb = RichProgressBar()

# 6⃣ Instantiate the Lightning Trainer
trainer = pl.Trainer(
    max_epochs=20,
    # compile=True,              # requires Lightning ≥2.0
    logger=logger,
    callbacks=[checkpoint_cb, earlystop_cb, progress_cb],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",            # **mixed-precision AMP**
    accumulate_grad_batches=2,  # 2× bigger "effective" batch
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    # compile=True,
)

Config.learning_rate = 1e-04

# 7⃣ Finally, run training & testing:
lit_model = LitLateFusion(
    learning_rate=Config.learning_rate,
    weight_decay=1e-4,
    unfreeze_last_n=3,
    scheduler_factor=Config.scheduler_factor,
    scheduler_patience=Config.patience,
    scheduler_threshold=Config.scheduler_threshold,
    min_lr=Config.min_lr,
    num_classes=len(answer2idx),
    weight_tensor = weight_tensor,
    idx2label=idx2answer
)

# lit_model = torch.compile(lit_model)

# trainer.fit(lit_model, train_loader, val_loader)


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


In [ ]:
import torch._dynamo
# globally suppress guard failures
torch._dynamo.config.suppress_errors = True

lit_model = torch.compile(lit_model)

In [ ]:
from pytorch_lightning.tuner import Tuner # import the Tuner class
import glob

# 3) Wrap your Trainer in a Tuner
tuner = Tuner(trainer)

# 4) Run the LR finder
lr_finder = tuner.lr_find(
    lit_model,
    train_dataloaders=train_loader,    # your existing DataLoader
    val_dataloaders=val_loader,        # ditto
    min_lr=1e-7,
    max_lr=1e-2,
    num_training=100,
    mode="exponential",
    early_stop_threshold=4.0
)

# 5) Inspect / choose the suggested LR
suggested_lr = lr_finder.suggestion()
print(f"👉 Suggested learning rate: {suggested_lr:.2e}")

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:652: Checkpoint directory /content/drive/MyDrive/tb_runs/early-swin exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.tuner.lr_finder:Learning rate set to 0.00011220184543019631
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_7a1bf287-a9c5-4a7a-ac15-491d403a5e72.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/.lr_find_7a1bf287-a9c5-4a7a-ac15-491d403a5e72.ckpt


👉 Suggested learning rate: 1.12e-04


In [ ]:
# trainer.fit(lit_model, train_loader, val_loader)
trainer.fit(lit_model, train_loader, val_loader, ckpt_path=best_ckpt)


/usr/local/lib/python3.11/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/tb_runs/early-swin exists and is not empty.
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/MyDrive/tb_runs/early-swin/early-swin-best-all-data-attention.ckpt
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                 ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ backbone  │ EarlyFusionSwinModel │  214 M │ train │
│ 1 │ criterion │ CrossEntropyLoss     │      0 │ train │
│ 2 │ train_acc │ MulticlassAccuracy   │      0 │ train │
│ 3 │ val_acc   │ MulticlassAccuracy   │      0 │ train │
└───┴───────────┴──────────────────────┴────────┴───────┘

Trainable params: 110 M                                                                                            
Non-trainable params: 103 M                                                                                        
Total params: 214 M                                                                                                
Total estimated model params size (MB): 857                                                                        
Modules in train mode: 12                                                                                          
Modules in eval mode: 711

INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/drive/MyDrive/tb_runs/early-swin/early-swin-best-all-data-attention.ckpt


Output()

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
trainer.test(lit_model, test_loader)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

🧾 TEST SET CLASSIFICATION REPORT

precision    recall  f1-score   support

       acoperit de nori       0.00      0.00      0.00         1
               acoperiş       0.50      1.00      0.67         1
                 adidas       0.00      0.00      0.00         3
                adidași       0.00      0.00      0.00         2
                  adânc       0.33      1.00      0.50         1
               aeroport       0.00      0.00      0.00         0
                  afişa       0.00      0.00      0.00         0
                 agitat       0.00      0.00      0.00         1
                    alb       0.31      0.32      0.31       112
              alb-negru       0.67      0.29      0.40         7
               albastru       0.34      0.49      0.40        63
       albastru deschis       0.50      0.22      0.31         9
        albastru inchis       0.00      0.00      0.00         4
                 albire       0.00      0.00      0.00         1
                 alcool       0.00      0.00      0.00         1
                   alee       0.00      0.00      0.00         3
               alergare       0.00      0.00      0.00         1
                  aluat       0.62      0.44      0.52        34
               aluminiu       1.00      1.00      1.00         3
              ambulanță       0.00      0.00      0.00         4
         animal impaiat       0.00      0.00      0.00         0
       animale impaiate       0.00      0.00      0.00         0
                  antic       0.00      0.00      0.00         0
               antrenor       0.00      0.00      0.00         1
            aparat foto       0.26      0.69      0.38        13
                    apă       0.00      0.00      0.00         2
                 aragaz       0.00      0.00      0.00         2
                arbitru       0.34      0.50      0.41        22
                 argint       0.41      0.38      0.39        32
             argintărie       0.00      0.00      0.00         2
                arzător       0.00      0.00      0.00         4
                arătând       0.00      0.00      0.00         1
                 asfalt       1.00      1.00      1.00         1
                asiatic       0.25      1.00      0.40         5
                  atlet       0.00      0.00      0.00        30
                    aur       0.00      0.00      0.00         8
                autobuz       0.56      0.97      0.71        36
             autocolant       0.00      0.00      0.00         2
                  avion       0.06      1.00      0.11         1
                avocado       0.00      0.00      0.00         0
              aşteptare       0.24      0.67      0.35         6
                 bagaje       0.00      0.00      0.00         3
                   baie       0.50      0.50      0.50         2
                  balon       0.00      0.00      0.00         0
                  baltă       0.00      0.00      0.00         0
                 banane       0.50      0.56      0.53         9
                 banană       0.71      1.00      0.83         5
                  bancă       0.27      0.40      0.32        15
                bandană       0.00      0.00      0.00         2
                  bandă       0.00      0.00      0.00         3
         barci cu pânze       0.00      0.00      0.00         7
                  barcă       0.06      1.00      0.12         1
         barcă cu pânze       0.31      0.71      0.43         7
               baseball       0.00      0.00      0.00         5
                   baut       0.00      0.00      0.00         0
                    bec       0.00      0.00      0.00         1
                    bej       1.00      0.11      0.20        18
     bentita pentru cap       0.00      0.00      0.00         2
                   bere       0.00      0.00      0.00         0
                  beton       1.00      0.50      0.67        28
                  bezea       0.00      0.00      0.00     

✅ Classification report saved to 
/content/drive/MyDrive/licenta/big_data/reports/test_report_early_big_data_swin.json

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/acc          │    0.4823501408100128     │
│         test/loss         │    2.6471779346466064     │
└───────────────────────────┴───────────────────────────┘

[{'test/loss': 2.6471779346466064, 'test/acc': 0.4823501408100128}]

In [ ]:
trainer.validate(lit_model, test_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/acc          │    0.42264270782470703    │
│         val/loss          │     2.822326898574829     │
└───────────────────────────┴───────────────────────────┘

[{'val/loss': 2.822326898574829, 'val/acc': 0.42264270782470703}]

In [ ]:
# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LateFusionModel().to(device)  # or LateFusionModel()
criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=1e-5)

# Add learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',  # Monitor accuracy
    factor=Config.scheduler_factor,
    patience=Config.patience,  # More frequent than early stopping
    threshold=Config.scheduler_threshold,
    min_lr=Config.min_lr,
    verbose=True
)

# Early stopping variables
best_acc = 0.0
epochs_no_improve = 0
early_stop = False


In [ ]:
from sklearn.metrics import accuracy_score
import torch

def sanity_overfit_on_batch(model, batch, device="cuda", num_iters=100, lr=3e-4):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.train()

    # Use only this batch
    batch = {k: v.to(device) for k, v in batch.items()}
    inputs = {k: v for k, v in batch.items() if k != "label"}
    labels = batch["label"]

    # Optimizer & loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()

    print("🚨 Sanity Overfit Test on One Batch")
    for i in range(1, num_iters + 1):
        optimizer.zero_grad()
        outputs = model(**inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean().item()

        print(f"[{i}/{num_iters}] Loss: {loss.item():.4f} | Acc: {acc:.4f}")


In [ ]:
# Grab a single batch from your train loader
single_batch = next(iter(train_loader))

# Run sanity check
sanity_overfit_on_batch(model, single_batch)


🚨 Sanity Overfit Test on One Batch
[1/100] Loss: 7.4604 | Acc: 0.0000
[2/100] Loss: 7.3988 | Acc: 0.0078
[3/100] Loss: 7.3234 | Acc: 0.0078
[4/100] Loss: 7.2157 | Acc: 0.0312
[5/100] Loss: 7.0961 | Acc: 0.0859
[6/100] Loss: 6.9188 | Acc: 0.1094
[7/100] Loss: 6.7462 | Acc: 0.1094
[8/100] Loss: 6.4861 | Acc: 0.1875
[9/100] Loss: 6.2336 | Acc: 0.1016
[10/100] Loss: 5.8422 | Acc: 0.1406
[11/100] Loss: 5.4805 | Acc: 0.0781
[12/100] Loss: 5.1172 | Acc: 0.0781
[13/100] Loss: 4.8348 | Acc: 0.0547
[14/100] Loss: 4.4487 | Acc: 0.1172
[15/100] Loss: 4.1748 | Acc: 0.2109
[16/100] Loss: 4.0081 | Acc: 0.2656
[17/100] Loss: 3.7211 | Acc: 0.2656
[18/100] Loss: 3.4571 | Acc: 0.3438
[19/100] Loss: 3.3525 | Acc: 0.4219
[20/100] Loss: 3.0022 | Acc: 0.6562
[21/100] Loss: 2.8392 | Acc: 0.5625
[22/100] Loss: 2.6263 | Acc: 0.7109
[23/100] Loss: 2.3933 | Acc: 0.7969
[24/100] Loss: 2.2522 | Acc: 0.7969
[25/100] Loss: 2.1490 | Acc: 0.8438
[26/100] Loss: 1.9228 | Acc: 0.8906
[27/100] Loss: 1.8460 | Acc: 0.8750
[2

KeyboardInterrupt: 

In [ ]:
# how many labels do you actually have?
n_labels = len(answer2idx)
print("Answer2idx size:", n_labels)

# what’s your model’s last‐layer out_features?
head = lit_model.backbone.classifier[-1]
print("Head out_features:", head.out_features)

assert head.out_features == n_labels, (
    f"Head dim ({head.out_features}) ≠ #labels ({n_labels})"
)


Answer2idx size: 1734
Head out_features: 1734


In [ ]:
batch = next(iter(train_loader))
imgs = batch["image"].to(device)
ids  = batch["input_ids"].to(device)
mask = batch["attention_mask"].to(device)
labs = batch["label"].to(device)

with torch.no_grad():
    logits = lit_model.backbone(imgs, ids, mask)
    preds  = logits.argmax(dim=1)

print("Batch labels:", torch.unique(labs))
print("Batch preds: ", torch.unique(preds))
print("Logits shape:", logits.shape)


Batch labels: tensor([   5,   27,   48,   58,   66,   79,   88,   90,  131,  145,  159,  161,
         177,  189,  196,  201,  202,  204,  208,  211,  213,  217,  223,  231,
         256,  261,  283,  287,  302,  322,  338,  339,  346,  347,  378,  385,
         394,  396,  426,  436,  440,  464,  519,  534,  558,  580,  607,  615,
         620,  623,  630,  646,  651,  691,  695,  702,  715,  721,  795,  813,
         818,  825,  828,  869,  872,  883,  913,  914,  917,  936,  952,  962,
         983,  985,  991, 1029, 1041, 1045, 1058, 1073, 1085, 1105, 1108, 1111,
        1119, 1162, 1171, 1183, 1200, 1204, 1209, 1276, 1278, 1279, 1301, 1303,
        1308, 1309, 1329, 1354, 1383, 1389, 1411, 1419, 1420, 1421, 1434, 1453,
        1458, 1489, 1490, 1492, 1532, 1541, 1554, 1563, 1585, 1625, 1626, 1653,
        1658, 1672, 1696, 1707], device='cuda:0')
Batch preds:  tensor([1724, 1725, 1727, 1728, 1729, 1730, 1731, 1732], device='cuda:0')
Logits shape: torch.Size([128, 1734])
